# Machine Learning #09 — Transformer Models in PyTorch 🤖

Today we meet the architecture behind **almost every modern language model**: the
**Transformer**. We will not build one from scratch — instead we will take a
**pre-trained** model, look inside it, and **fine-tune** it for our own task.

Everything runs in [PyTorch](https://pytorch.org/) together with the
🤗 [Hugging Face](https://huggingface.co/) libraries.

### Our workflow today

> **text → tokens → pre-trained model → embeddings & attention → fine-tuning → predictions**

##### ⚠️ Remember to set the **GPU** runtime in Colab! (`Runtime → Change runtime type → T4 GPU`)

# 📒 What is a Transformer?

The core idea is **self-attention**: each token combines information from other allowed
positions in its context. An encoder can use both directions; a causal decoder cannot
attend to future tokens.

Take the sentence:

> *The animal didn't cross the street because **it** was too tired.*

Representing *it* may require information about *the animal*. Self-attention provides
a direct route between those positions, but a single attention map is not proof that
the model resolved the reference correctly.

## ⚙️ Self-attention in one formula

Learned projections of the same input sequence produce **queries** $Q$, **keys** $K$
and **values** $V$; the three matrices need not be equal.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

- $QK^\top$ contains a query–key score for every pair of positions.
- $\sqrt{d_k}$ controls the score scale as the query/key dimension grows.
- Softmax runs over the **key** axis: each query's weights sum to one before dropout.
- Multiplying by $V$ gives a weighted sum of the value vectors.

**Multi-head attention** uses several learned projections in parallel, concatenates
the results and projects them again. Heads can learn different patterns, without a
guaranteed linguistic role for each head. Transformer blocks also contain a feed-forward
network, residual connections and normalization; attention is one part of the block.

## 🔎 How is this different from an RNN?

| | RNN / LSTM | Transformer |
|---|---|---|
| Sequence processing | recurrent steps depend on previous steps | attention can process positions in parallel during training |
| Context | information passes through recurrent states | direct connections to allowed positions within the context window |
| Long dependencies | require many recurrent steps | can connect in one attention step |
| Word order | encoded by the recurrence | supplied through position information |

DistilBERT adds **learned positional embeddings** to token embeddings. Other transformers
may use relative or rotary position representations. Autoregressive generation still
produces new tokens sequentially; parallel training does not remove that dependency.

## 🏗️ Three common transformer families

| Family | Context | Typical tasks | Examples |
|---|---|---|---|
| **Encoder** | both directions | classification, NER, representations | BERT, DistilBERT, RoBERTa |
| **Decoder** | previous tokens and the current position | next-token generation | GPT-style models, Llama |
| **Encoder–decoder** | bidirectional input encoder and causal output decoder with cross-attention | translation, summarization | T5, BART |

Today we use an **encoder** for classification.

## 🎓 Pre-training → fine-tuning

1. **Pre-training** learns reusable representations from a large corpus. BERT-style
   encoders predict masked tokens; causal language models predict the next token.
   DistilBERT also learns from a larger teacher model through **distillation**.
2. **Fine-tuning** adapts those representations and a task-specific classification head
   using labelled examples. Training time depends on hardware, sequence length and data size.

## 🐣 Our model: DistilBERT

We use [`distilbert/distilbert-base-uncased`](https://huggingface.co/distilbert/distilbert-base-uncased):

- a distilled version of BERT with **6 layers**, **12 heads per layer**, hidden size **768**
  and approximately **66 million parameters**,
- its original study reported about **40% fewer parameters** and **60% faster inference**
  than BERT-base under the study's conditions; speedups depend on the workload,
- *uncased* means its tokenizer lowercases text and strips accents, so case distinctions
  are lost before the model sees the input.

## ⚙️ Imports and setup

We need two Hugging Face libraries:

- **`transformers`** — the models and tokenizers themselves,
- **`datasets`** — a convenient loader for benchmark datasets.

The installation cell uses `%pip` so packages go into the current notebook kernel.
The same dependencies are declared in the repository's `pyproject.toml`.

Everything else (`torch`, `sklearn`, `matplotlib`) you already know.

In [ ]:
%pip install -q "transformers>=4.41" "datasets>=2.19"

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import DataLoader

from datasets import load_dataset

from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report, matthews_corrcoef
)
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch version:", torch.__version__)
print("device       :", device)

In [ ]:
# The pre-trained checkpoint we will use through the whole notebook.
# A "checkpoint" is simply a name of a model stored on the Hugging Face Hub.
CHECKPOINT = "distilbert/distilbert-base-uncased"

# 🚀 Part 1 — A transformer in three lines

Before we dissect anything, let's see what these models can do out of the box.

The **`pipeline`** helper wraps the *whole* chain — tokenizer → model → post-processing —
into a single callable object. Here we use a DistilBERT that someone has **already
fine-tuned** for sentiment analysis.

> 💡 The first run downloads model files of roughly 250 MB. Later loads reuse the
> local cache; download and loading times depend on the connection and hardware.

In [ ]:
# task name  +  model checkpoint  ->  ready-to-use predictor
sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1,   # 0 = first GPU, -1 = CPU
)

In [ ]:
sentiment("I absolutely loved this movie, the acting was brilliant!")

In [ ]:
# The pipeline also accepts a list of texts and returns one dict per text
reviews = [
    "The plot was predictable and the characters were flat.",
    "A masterpiece. I would watch it again tomorrow.",
    "It was okay, nothing special.",
]

for text, result in zip(reviews, sentiment(reviews)):
    print(f"{result['label']:>8}  ({result['score']:.3f})  |  {text}")

### 🤔 What just happened?

The pipeline loaded an already fine-tuned classifier, tokenized our text and converted
its logits into labels and scores. Its [model card](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english)
reports **91.3% accuracy on the SST-2 development set**. That is a benchmark result, not
a guarantee for arbitrary reviews or other domains.

The score is the model's probability for its predicted class; it need not be calibrated
to the actual probability of being correct. Next we inspect the components behind the pipeline.

## ✏️ Activity 1 — Break the model 💥

Sentiment models are famously bad at **sarcasm**, **negation** and **mixed opinions**.

**Your task:** find at least **two** sentences where the model is confidently *wrong*,
and one where it is *unsure* (score close to 0.5).

In [ ]:
my_sentences = [
    # Add your examples here.
]

if my_sentences:
    for text, result in zip(my_sentences, sentiment(my_sentences, truncation=True)):
        print(f"{result['label']:>8}  ({result['score']:.3f})  |  {text}")
else:
    print("Add sentences to my_sentences, then rerun this cell.")

# 🔤 Part 2 — The tokenizer

A neural network cannot read text — it needs **numbers**. The **tokenizer** does that
conversion, and it is *part of the model*: every checkpoint ships with the exact
tokenizer it was trained with.

`AutoTokenizer.from_pretrained` downloads the right one automatically.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, use_fast=True)
tokenizer

In [ ]:
text = "Transformers are changing machine learning."

encoded = tokenizer(text)
encoded

### What is inside the output?

| Key | Meaning |
|---|---|
| `input_ids` | the **id of every token** in the model's vocabulary |
| `attention_mask` | `1` = a real token, `0` = padding that must be **ignored** |

Let's translate the ids back into readable tokens.

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
tokens

### 🎁 Special tokens

Two tokens were added that we never wrote:

- **`[CLS]`** — first token for this tokenizer. DistilBERT's classification head reads
  its final hidden state. Fine-tuning teaches it to provide information for that task;
  it is not automatically a general-purpose sentence embedding.
- **`[SEP]`** — *separator*, marks the end of a sentence (or the boundary between two
  sentences, e.g. for question–answer pairs).

In [ ]:
print("Vocabulary size :", tokenizer.vocab_size)
print("Special tokens  :", tokenizer.all_special_tokens)
print("[CLS] id        :", tokenizer.cls_token_id)
print("[SEP] id        :", tokenizer.sep_token_id)
print("[PAD] id        :", tokenizer.pad_token_id)

### 🧩 Subword tokenization

The vocabulary has only ~30 000 entries, but English has far more words — and users
invent new ones all the time. The trick: **rare words are split into pieces**
(*subwords*). The prefix `##` means *"this piece continues the previous token"*.

This represents many unseen words by combining known pieces. WordPiece can still
produce `[UNK]` for unsupported text; it cannot encode every possible word losslessly.

In [ ]:
weird = "Tokenization of unbelievably hyperparametrized transformers is unquestionable."

print(tokenizer.tokenize(weird))

In [ ]:
# Number of words vs. number of tokens
print("words :", len(weird.split()))
print("tokens:", len(tokenizer.tokenize(weird)))

> 💡 Context limits count **tokens**, including special tokens, rather than words.
> There is no fixed word-to-token ratio: it depends on the tokenizer, language and text.
> Compare the English and Czech examples below using this English tokenizer.

In [ ]:
czech = "Transformátorové modely změnily strojové učení."
print(tokenizer.tokenize(czech))
print("words :", len(czech.split()), "| tokens:", len(tokenizer.tokenize(czech)))

### 📦 Batches: padding and truncation

Sentences have different lengths, but a tensor must be **rectangular**. So we:

- **pad** shorter sentences with `[PAD]` up to the longest one in the batch,
- **truncate** at a chosen maximum length, which must not exceed the checkpoint's
  context limit (512 tokens for our DistilBERT, including special tokens).

The example below deliberately uses the smaller limit **32**.

The `attention_mask` then tells the model *"ignore these padded positions"*.

In [ ]:
batch_texts = [
    "Short one.",
    "This sentence is considerably longer than the previous one.",
]

batch = tokenizer(
    batch_texts,
    padding=True,            # pad to the longest sequence in the batch
    truncation=True,         # cut anything above max_length
    max_length=32,
    return_tensors="pt",     # "pt" = PyTorch tensors (this is the PyTorch world 🔥)
)

batch

In [ ]:
# Both sequences now have exactly the same shape
batch["input_ids"].shape

In [ ]:
# Look at the padded sentence: [PAD] tokens have attention_mask = 0
for tok, mask in zip(tokenizer.convert_ids_to_tokens(batch["input_ids"][0]),
                     batch["attention_mask"][0].tolist()):
    print(f"{tok:>12}  ->  mask = {mask}")

In [ ]:
# And back from ids to text
tokenizer.decode(batch["input_ids"][0], skip_special_tokens=True)

# 🧠 Part 3 — Inside the model

There are two ways to load a transformer:

| Class | What you get | Use it for |
|---|---|---|
| `AutoModel` | the **backbone** only — a vector for every token | embeddings, feature extraction, exploration |
| `AutoModelForSequenceClassification` | backbone **+ a small classification head** | classification, fine-tuning |

Let's start with the bare backbone to see what a transformer actually outputs.

In [ ]:
backbone = AutoModel.from_pretrained(CHECKPOINT).to(device)
backbone.eval()          # evaluation mode: no dropout (we are not training yet)
print(backbone.config)

In [ ]:
n_params = sum(p.numel() for p in backbone.parameters())
print(f"Number of parameters: {n_params:,}")

In [ ]:
# The architecture: embeddings + 6 transformer blocks
backbone

## 📐 What comes out of a transformer?

We push a sentence through the model. We use
**`torch.no_grad()`** — we are only doing inference, so no gradients are needed
(it is faster and saves memory).

In [ ]:
sentence = "The cat sat on the mat."

inputs = tokenizer(sentence, return_tensors="pt").to(device)   # don't forget .to(device)!

with torch.no_grad():
    outputs = backbone(**inputs)     # ** unpacks input_ids and attention_mask as arguments

outputs.last_hidden_state.shape

### Reading the shape: `(batch_size, sequence_length, hidden_size)`

- **1** — one sentence in the batch
- **9** — number of tokens (`[CLS]` + 7 tokens + `[SEP]`)
- **768** — the size of the vector representing **each token**

So the model does *not* return one vector per sentence — it returns
**one 768-dimensional vector per token**. 🎯

In [ ]:
print("tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
print("shape :", outputs.last_hidden_state.shape)

# The vector of the very first token ([CLS]) - just the first 10 numbers
outputs.last_hidden_state[0, 0, :10]

## ✨ Contextual embeddings — the key idea

In classic embeddings (word2vec, GloVe) every word has **one fixed vector**.
The word *bank* is the same vector in *river bank* and in *investment bank*.

A transformer builds the vector **from the context** — so the same word gets
**different vectors in different sentences**. Let's prove it.

In [ ]:
def token_embedding(sentence, word, occurrence=0):
    """Average the subword states of one word occurrence (0 = first occurrence)."""
    if not tokenizer.is_fast:
        raise ValueError("This example needs a fast tokenizer for character offsets")
    if not word or occurrence < 0:
        raise ValueError("Provide a nonempty word and a nonnegative occurrence index")
    matches = list(re.finditer(r"(?<!\w)" + re.escape(word) + r"(?!\w)",
                               sentence, flags=re.IGNORECASE))
    if occurrence >= len(matches):
        raise ValueError(f"Occurrence {occurrence} of {word!r} was not found")
    start, end = matches[occurrence].span()

    enc = tokenizer(sentence, truncation=True, return_offsets_mapping=True,
                    return_tensors="pt")
    offsets = enc.pop("offset_mapping")[0].tolist()
    positions = [i for i, (a, b) in enumerate(offsets) if a < end and b > start]
    if (not positions or offsets[positions[0]][0] > start
            or offsets[positions[-1]][1] < end):
        raise ValueError("The selected word was removed or cut by truncation")

    enc = enc.to(next(backbone.parameters()).device)
    backbone.eval()
    with torch.no_grad():
        hidden = backbone(**enc).last_hidden_state
    return hidden[0, positions].mean(dim=0).cpu().numpy()

In [ ]:
sent_a = "I deposited the money at the bank yesterday."     # financial
sent_b = "The savings bank raised its interest rates."      # financial
sent_c = "We sat on the river bank and watched the sunset." # geographical

vectors = np.stack([
    token_embedding(sent_a, "bank"),
    token_embedding(sent_b, "bank"),
    token_embedding(sent_c, "bank"),
])

vectors.shape

In [ ]:
labels = ["bank (money A)", "bank (money B)", "bank (river)"]
sim = cosine_similarity(vectors)

plt.figure(figsize=(5, 4))
sns.heatmap(sim, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            xticklabels=labels, yticklabels=labels, vmin=-1, vmax=1)
plt.title("Cosine similarity of the word 'bank'")
plt.tight_layout()
plt.show()

> 💡 The same word receives different vectors in different contexts. Inspect the actual
> similarities: the two financial uses may be closer, but this ordering is not guaranteed
> for every example or layer. A few vectors do not establish reliable word-sense resolution.

## 🧵 From tokens to one sentence vector

Two common ways to obtain a fixed-size vector are:

1. **`[CLS]` pooling** — use the first token's hidden state.
2. **Masked mean pooling** — average non-padding states. Our version includes `[CLS]`
   and `[SEP]`, since their attention-mask values are also 1.

We demonstrate mean pooling here. Neither pooling method turns a masked-language-model
checkpoint into a model specifically trained for sentence similarity. Such models usually
need an appropriate sentence-level training objective; compare pooling methods empirically.

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    # attention_mask: (batch, seq) -> (batch, seq, 1) so it broadcasts over the 768 dims
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)   # sum only real tokens
    counts = mask.sum(dim=1)                         # how many real tokens
    return summed / counts.clamp_min(1)


def sentence_embeddings(sentences):
    if isinstance(sentences, str):
        sentences = [sentences]
    if not sentences:
        raise ValueError("Provide at least one sentence")
    enc = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
    enc = enc.to(next(backbone.parameters()).device)
    backbone.eval()
    with torch.no_grad():
        hidden = backbone(**enc).last_hidden_state
    return mean_pool(hidden, enc["attention_mask"]).cpu().numpy()

In [ ]:
sentences = [
    "The weather is beautiful today.",
    "It is a lovely sunny day.",
    "My laptop crashed again during the exam.",
    "The computer keeps freezing when I work.",
    "She adopted a small orange kitten.",
]

emb = sentence_embeddings(sentences)
print("embeddings shape:", emb.shape)      # (5 sentences, 768)

In [ ]:
sim = cosine_similarity(emb)
short = [s[:28] + "..." for s in sentences]

plt.figure(figsize=(7, 5.5))
sns.heatmap(sim, annot=True, fmt=".2f", cmap="magma",
            xticklabels=short, yticklabels=short)
plt.title("Cosine similarity of mean-pooled DistilBERT states")
plt.tight_layout()
plt.show()

> 🔎 Do the weather sentences and the computer sentences form the pairs you expected?
> These are exploratory similarities from a pretrained model, with no additional training
> here. Strong performance in semantic search would need evaluation on representative data;
> this small heatmap alone does not establish it.

# 👀 Part 4 — Looking inside attention

For DistilBERT with a compatible implementation, **`output_attentions=True`** returns
attention matrices for each layer and head. In evaluation mode, dropout is disabled and
the rows correspond to normalized query–key weights.

## ⚠️ Attention implementations

PyTorch's **SDPA** dispatches to different implementations depending on the device, dtype
and tensor shapes. Efficient fused kernels can avoid materializing the full attention
matrix, but SDPA is not synonymous with FlashAttention or a particular GPU kernel.

The SDPA interface returns attention **outputs**, not the weight matrices. Depending on
the Transformers model and version, requesting weights may trigger a fallback or be
unsupported. We explicitly load **`attn_implementation="eager"`** for this demonstration
so that attention weights are available.

See the [Hugging Face attention documentation](https://huggingface.co/docs/transformers/en/attention_interface).

> 💡 A large attention weight shows how a head mixes value vectors. It does not, by itself,
> prove that a token caused the final prediction or that a head learned a particular rule.

In [ ]:
# Load the same checkpoint again, but with the "eager" attention implementation
# so that the attention matrices are actually computed and returned.
# Cached weights avoid another download, but loading still takes time and memory.
backbone_eager = AutoModel.from_pretrained(
    CHECKPOINT,
    attn_implementation="eager",     # <- the important part
).to(device)

backbone_eager.eval()

print("attention implementation:", backbone_eager.config._attn_implementation)

### Reading the shape: `(batch, heads, seq_len, seq_len)`

- **6 layers** × **12 heads** = **72 attention matrices** for a single sentence!
- each matrix is `seq_len × seq_len`: row *i* tells us **how much token *i* attends to
  every other token**
- each row sums to approximately 1 in **evaluation mode**, up to numerical rounding

In [ ]:
# Helper: run a sentence through the model and return its attention maps + tokens
def get_attention(sentence):
    enc = tokenizer(sentence, truncation=True, return_tensors="pt")
    enc = enc.to(next(backbone_eager.parameters()).device)
    backbone_eager.eval()

    with torch.no_grad():
        out = backbone_eager(**enc, output_attentions=True)

    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    if out.attentions is None or not out.attentions:
        raise RuntimeError("Attention weights are unavailable; load a supported model in eager mode")
    return tuple(weights.cpu() for weights in out.attentions), tokens


# Plot the attention matrix of one head in one layer.
# attentions + tokens come from get_attention(); head=None averages all available heads.
def plot_attention(attentions, tokens, layer, head=0, title=None):
    if head is None:
        matrix = attentions[layer][0].mean(dim=0).cpu().numpy()   # average over heads
        head_name = "mean of all heads"
    else:
        matrix = attentions[layer][0, head].cpu().numpy()          # (seq_len, seq_len)
        head_name = f"head {head}"

    plt.figure(figsize=(8, 6.5))
    sns.heatmap(matrix, xticklabels=tokens, yticklabels=tokens,
                cmap="Blues", vmin=0, square=True, cbar_kws={"shrink": 0.7})
    plt.title(title or f"Attention — layer {layer}, {head_name}")
    plt.ylabel("query token (who is looking)")
    plt.xlabel("key token (where it looks)")
    plt.tight_layout()
    plt.show()

In [ ]:
att_sentence = "The animal did not cross the street because it was too tired."
att, tokens = get_attention(att_sentence)
plot_attention(att, tokens, layer=min(4, len(att) - 1),
               head=min(6, att[0].shape[1] - 1))

In [ ]:
def plot_all_heads(attentions, tokens, layer):
    # Derive the grid from the model rather than assuming exactly 12 heads.
    n_heads = attentions[layer].shape[1]
    n_cols = min(4, n_heads)
    n_rows = (n_heads + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows), squeeze=False)

    for head, ax in enumerate(axes.flat):
        if head >= n_heads:
            ax.set_visible(False)
            continue
        sns.heatmap(attentions[layer][0, head].cpu().numpy(), ax=ax,
                    cmap="Blues", vmin=0, cbar=False, square=True,
                    xticklabels=tokens, yticklabels=tokens)
        ax.set_title(f"head {head}", fontsize=10)
        ax.tick_params(labelsize=7)

    fig.suptitle(f"All {n_heads} attention heads — layer {layer}", fontsize=15)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_all_heads(att, tokens, layer=min(3, len(att) - 1))

# 🎯 Part 5 — Fine-tuning #1: sentiment (SST-2)

Time to **train**. We take the pre-trained DistilBERT, attach a fresh classification
head, and teach it a task from the [GLUE benchmark](https://gluebenchmark.com/) —
a standard collection of nine English language-understanding datasets.

### Our task: SST-2 (Stanford Sentiment Treebank)

- **Input:** one sentence (a fragment of a movie review)
- **Output:** `0` = negative, `1` = positive
- **Metric:** accuracy

| GLUE task | What it decides |
|---|---|
| **`sst2`** | sentiment of a sentence ← *we start here* |
| **`cola`** | is the sentence grammatically acceptable? ← *we do this one too* |
| `mrpc` | are two sentences paraphrases? |
| `qnli` | does a sentence answer a question? |
| `rte`, `mnli` | does one sentence entail another? |
| `stsb` | how similar are two sentences (regression) |
| `qqp` | are two Quora questions duplicates? |

In [ ]:
raw_sst2 = load_dataset("nyu-mll/glue", "sst2")
raw_sst2

In [ ]:
# One example: the text, the label and an index
raw_sst2["train"][0]

In [ ]:
# What do the label numbers mean?
raw_sst2["train"].features["label"]

In [ ]:
# Class balance - are the two classes roughly equally represented?
sst2_labels = pd.Series(raw_sst2["train"]["label"]).map({0: "negative", 1: "positive"})

plt.figure(figsize=(5, 3))
sns.countplot(x=sst2_labels)
plt.title("SST-2 — class balance")
plt.tight_layout()
plt.show()

print(sst2_labels.value_counts(normalize=True).round(3))

### 🐢 Keeping the lab fast

SST-2 has roughly 67,000 training examples. We use a shuffled subset to keep the exercise
manageable; more examples can still improve performance. Runtime depends on the hardware,
model, batch size and token length.

We use GLUE's labelled **validation** split for checkpoint selection and reporting.
The public test labels are hidden, so the scores below are development-set results,
not independent test scores. Repeated tuning can overfit this validation set.

In [ ]:
N_TRAIN = 4000          # change this in Activity 4
MAX_LEN = 64            # a speed/coverage tradeoff; longer examples are truncated
BATCH_SIZE = 32

def training_subset(dataset, n_samples, seed=SEED):
    if n_samples < 1:
        raise ValueError("n_samples must be positive")
    # Larger requests use the available data instead of indexing beyond the dataset.
    return dataset.shuffle(seed=seed).select(range(min(n_samples, len(dataset))))

sst2_train_raw = training_subset(raw_sst2["train"], N_TRAIN)
sst2_eval_raw = raw_sst2["validation"]

print("training examples:", len(sst2_train_raw), "| validation examples:", len(sst2_eval_raw))

## 🧹 Preprocessing: tokenize the whole dataset

`dataset.map()` applies our tokenizing function to **every example** (in batches, so it
is fast). Then we keep only the columns the model expects.

> ⚠️ The label column **must be named `labels`** — the model looks for exactly this name.

In [ ]:
def encode(dataset, text_key, max_length=None):
    max_length = MAX_LEN if max_length is None else max_length
    # 1) tokenize (no padding here - the collator will pad each batch separately)
    def tok(batch):
        return tokenizer(batch[text_key], truncation=True, max_length=max_length)

    encoded = dataset.map(tok, batched=True)

    # 2) drop everything the model does not need (raw text, idx, ...)
    keep = set(tokenizer.model_input_names) | {"label"}  # includes token_type_ids when needed
    encoded = encoded.remove_columns([c for c in encoded.column_names if c not in keep])

    # 3) rename label -> labels and return PyTorch tensors
    encoded = encoded.rename_column("label", "labels")
    encoded.set_format("torch")
    return encoded

In [ ]:
sst2_train = encode(sst2_train_raw, "sentence")
sst2_eval  = encode(sst2_eval_raw,  "sentence")

sst2_train

In [ ]:
sst2_train[0]

### 🚚 DataLoader + dynamic padding

We use a **`DataLoader`**, which serves the data in
**mini-batches**.

`DataCollatorWithPadding` pads each batch to the length of *its own* longest sequence
(**dynamic padding**). Padding everything to 512 tokens would waste a lot of compute.

In [ ]:
collator = DataCollatorWithPadding(tokenizer=tokenizer)

sst2_train_loader = DataLoader(
    sst2_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator,
    generator=torch.Generator().manual_seed(SEED),
)
sst2_eval_loader = DataLoader(sst2_eval, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)

print("training batches:", len(sst2_train_loader))

In [ ]:
# Peek at one batch - notice the shapes
example_batch = next(iter(sst2_train_loader))
{k: v.shape for k, v in example_batch.items()}

## 🏗️ The model with a classification head

`AutoModelForSequenceClassification` loads a backbone and a task head. For **DistilBERT**
the classification path is:

$$\text{[CLS] vector (768)} \;\rightarrow\; \text{Linear} \;\rightarrow\; \text{ReLU} \;\rightarrow\; \text{Dropout} \;\rightarrow\; \text{Linear} \;\rightarrow\; \text{2 logits}$$

The backbone weights come from pre-training, but the **head is brand new and random**.
Hugging Face warns you about exactly that — the message below is **expected, not an error**. 😉

In [ ]:
id2label = {0: "negative", 1: "positive"}
label2id = {v: k for k, v in id2label.items()}

torch.manual_seed(SEED)  # reproducible initialization of the new head
model_sst2 = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
).to(device)

In [ ]:
# Inspect modules outside the backbone without assuming a DistilBERT-only attribute.
{name: module for name, module in model_sst2.named_children()
 if name != model_sst2.base_model_prefix}

## 🧪 Baseline: how good is the model *before* training?

Measure the random-head baseline. Accuracy can differ substantially from 50% in one
initialization because predictions need not be balanced or independent of the inputs.
Also compare it with the accuracy of always predicting the training-set majority class.

In [ ]:
def predict(model, loader):
    model.eval()                       # evaluation mode
    logits_all, labels_all = [], []

    with torch.no_grad():              # no gradients -> faster, less memory
        for batch in loader:
            labels_all.append(batch["labels"].cpu().numpy())
            inputs = {k: v.to(next(model.parameters()).device)
                      for k, v in batch.items() if k != "labels"}
            logits_all.append(model(**inputs).logits.cpu().numpy())

    if not logits_all:
        raise ValueError("The evaluation loader is empty")
    return np.concatenate(logits_all), np.concatenate(labels_all)

In [ ]:
logits, y_true = predict(model_sst2, sst2_eval_loader)
y_pred = logits.argmax(axis=1)         # highest logit wins

sst2_majority = int(np.bincount(np.asarray(sst2_train_raw["label"])).argmax())
majority_preds = np.full_like(y_true, sst2_majority)
print(f"Validation accuracy BEFORE fine-tuning: {accuracy_score(y_true, y_pred):.3f}")
print(f"Training-majority baseline accuracy   : {accuracy_score(y_true, majority_preds):.3f}")

## ⭐ The training loop

The model computes cross-entropy from its logits and the integer `labels`.
We average validation loss **per example**, including the smaller last batch, and restore
the checkpoint with the lowest validation loss. This selection rule is used for both tasks;
we report SST-2 accuracy and CoLA MCC separately afterward.

In [ ]:
from tqdm.auto import tqdm


def train_model(model, train_loader, val_loader, epochs=2, lr=2e-5, seed=SEED):
    if epochs < 1 or len(train_loader) == 0 or len(val_loader) == 0:
        raise ValueError("Use positive epochs and nonempty training/validation loaders")
    torch.manual_seed(seed)
    if train_loader.generator is not None:
        train_loader.generator.manual_seed(seed)
    model_device = next(model.parameters()).device
    trainable = [p for p in model.parameters() if p.requires_grad]
    if not trainable:
        raise ValueError("The model has no trainable parameters")
    optimizer = torch.optim.AdamW(trainable, lr=lr)

    total_steps = epochs * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps
    )
    history = {"step_loss": [], "val_loss": []}
    best_loss, best_state, best_epoch = float("inf"), None, None
    frozen_backbone = not any(p.requires_grad for p in model.base_model.parameters())

    for epoch in range(epochs):
        model.train()                         # trainable head keeps its dropout
        if frozen_backbone:
            model.base_model.eval()           # fixed features: disable backbone dropout

        for batch in tqdm(train_loader, desc=f"epoch {epoch + 1}/{epochs}"):
            batch = {k: v.to(model_device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            loss = model(**batch).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
            scheduler.step()
            history["step_loss"].append(loss.item())

        model.eval()
        loss_sum, n_examples = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(model_device) for k, v in batch.items()}
                batch_size = batch["labels"].size(0)
                loss_sum += model(**batch).loss.item() * batch_size
                n_examples += batch_size

        val_loss = loss_sum / n_examples
        history["val_loss"].append(val_loss)
        print(f"  -> epoch {epoch + 1}: validation loss = {val_loss:.4f}")
        if val_loss < best_loss:
            best_loss, best_epoch = val_loss, epoch + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is None:
        raise RuntimeError("Training did not produce a finite validation loss")
    model.load_state_dict(best_state)
    model.eval()
    optimizer.zero_grad(set_to_none=True)
    print(f"Restored epoch {best_epoch} with validation loss {best_loss:.4f}")
    return history

In [ ]:
%%time
history_sst2 = train_model(model_sst2, sst2_train_loader, sst2_eval_loader, epochs=2, lr=2e-5)

In [ ]:
steps = np.arange(len(history_sst2["step_loss"]))
rolling = pd.Series(history_sst2["step_loss"]).rolling(20, min_periods=1).mean()

plt.figure(figsize=(7, 4))
plt.plot(steps, history_sst2["step_loss"], alpha=0.25, label="train loss (per step)")
plt.plot(steps, rolling, label="train loss (rolling mean)")

# validation loss is measured once per epoch -> draw it at the end of each epoch
steps_per_epoch = len(sst2_train_loader)
for i, vl in enumerate(history_sst2["val_loss"]):
    plt.scatter((i + 1) * steps_per_epoch - 1, vl, marker="o", s=60, zorder=5,
                label="validation loss" if i == 0 else None)

plt.xlabel("training step")
plt.ylabel("cross-entropy loss")
plt.title("SST-2 fine-tuning")
plt.legend()
plt.tight_layout()
plt.show()

## 📊 Validation results

These scores use the same validation split that selected the checkpoint. They are useful
development results, but should not be presented as an untouched test-set evaluation.

In [ ]:
logits, y_true = predict(model_sst2, sst2_eval_loader)
y_pred = logits.argmax(axis=1)

print(f"Validation accuracy AFTER fine-tuning: {accuracy_score(y_true, y_pred):.3f}")
print()
print(classification_report(y_true, y_pred, labels=[0, 1], target_names=["negative", "positive"], zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

plt.figure(figsize=(4.5, 3.8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["negative", "positive"], yticklabels=["negative", "positive"])
plt.xlabel("predicted"); plt.ylabel("true")
plt.title("SST-2 — confusion matrix")
plt.tight_layout(); plt.show()

## 🔮 Inference on our own sentences

The same recipe as before: tokenize → forward pass → `softmax` → `argmax`.
Here we apply softmax to show the model's **class probabilities**. A high score can
still accompany an incorrect prediction; these probabilities are not necessarily calibrated.

In [ ]:
def classify(model, sentences, max_length=None):
    if isinstance(sentences, str):
        sentences = [sentences]
    sentences = list(sentences)
    if not sentences:
        return pd.DataFrame(columns=["sentence", "prediction", "confidence"])
    max_length = MAX_LEN if max_length is None else max_length
    enc = tokenizer(sentences, padding=True, truncation=True,
                    max_length=max_length, return_tensors="pt").to(next(model.parameters()).device)

    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits

    probs = torch.softmax(logits, dim=-1).cpu().numpy()   # logits -> probabilities
    preds = probs.argmax(axis=1)

    return pd.DataFrame({
        "sentence": sentences,
        "prediction": [model.config.id2label[int(p)] for p in preds],
        "confidence": probs.max(axis=1).round(3),
    })

In [ ]:
my_reviews = [
    "The soundtrack alone is worth the ticket.",
    "I have never been so bored in a cinema.",
    "A slow start, but the ending completely won me over.",
]

classify(model_sst2, my_reviews)

# 📏 Part 6 — Fine-tuning #2: grammar (CoLA)

Now the same machinery on a **very different** task — and one that will teach us
something about **metrics**.

### CoLA — Corpus of Linguistic Acceptability

- **Input:** one sentence
- **Output:** `1` = grammatically acceptable, `0` = not acceptable
- **Primary benchmark metric:** Matthews correlation coefficient (MCC); also inspect accuracy and per-class results.

The interesting part: sentiment can often be guessed from single words
(*brilliant*, *boring*), but grammar depends on **word order and structure** —
information that attention and position representations can help model.

In [ ]:
raw_cola = load_dataset("nyu-mll/glue", "cola")
raw_cola

In [ ]:
raw_cola["train"][:5]

In [ ]:
# Class balance - this time it is NOT balanced!
cola_labels = pd.Series(raw_cola["train"]["label"]).map({0: "unacceptable", 1: "acceptable"})

plt.figure(figsize=(5, 3))
sns.countplot(x=cola_labels)
plt.title("CoLA — class balance")
plt.tight_layout(); plt.show()

print(cola_labels.value_counts(normalize=True).round(3))

## ⚠️ Why accuracy is a trap here

About **70 %** of the sentences are acceptable. So a "model" that ignores the input and
always answers *acceptable* scores **70 % accuracy** — while being completely useless.

The official CoLA metric is therefore the **Matthews correlation coefficient (MCC)**:

$$\text{MCC} = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

| MCC | Meaning |
|---|---|
| $+1$ | perfect prediction |
| $0$ | no correlation between predictions and labels |
| $-1$ | perfectly wrong |

MCC uses **all four cells** of the confusion matrix. For constant predictions the
denominator is zero; scikit-learn returns **0** by convention. Independent random
predictions tend toward zero MCC, but a finite random sample need not score exactly zero.
Let's compare the majority-class baseline using both metrics.

In [ ]:
y_val_cola = np.asarray(raw_cola["validation"]["label"])

# Choose the constant class from TRAINING labels, then evaluate on validation labels.
cola_majority = int(np.bincount(np.asarray(raw_cola["train"]["label"])).argmax())
dumb = np.full_like(y_val_cola, cola_majority)

print(f"Majority baseline validation accuracy: {accuracy_score(y_val_cola, dumb):.3f}")
print(f"Majority baseline validation MCC     : {matthews_corrcoef(y_val_cola, dumb):.3f}")

> 🎯 Same predictions, two very different stories. **Choosing the metric is part of
> designing the experiment**, not an afterthought.

## 🚀 Training — exactly the same code

Notice how little changes: a different dataset and label meanings; both tasks use
the same `sentence` text-column name.
The tokenizer, the loop, the evaluation — all reused.

In [ ]:
cola_train_raw = training_subset(raw_cola["train"], N_TRAIN)
cola_eval_raw  = raw_cola["validation"]

cola_train = encode(cola_train_raw, "sentence")   # CoLA also uses the column "sentence"
cola_eval  = encode(cola_eval_raw,  "sentence")

cola_train_loader = DataLoader(
    cola_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator,
    generator=torch.Generator().manual_seed(SEED),
)
cola_eval_loader  = DataLoader(cola_eval,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)

len(cola_train), len(cola_eval)

In [ ]:
# A fresh model - we must NOT continue training the sentiment one!
torch.manual_seed(SEED)
model_cola = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=2,
    id2label={0: "unacceptable", 1: "acceptable"},
    label2id={"unacceptable": 0, "acceptable": 1},
).to(device)

In [ ]:
%%time
history_cola = train_model(model_cola, cola_train_loader, cola_eval_loader, epochs=3, lr=2e-5)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history_cola["step_loss"], alpha=0.25, label="train loss (per step)")
plt.plot(pd.Series(history_cola["step_loss"]).rolling(20, min_periods=1).mean(),
         label="train loss (rolling mean)")

steps_per_epoch = len(cola_train_loader)
for i, vl in enumerate(history_cola["val_loss"]):
    plt.scatter((i + 1) * steps_per_epoch - 1, vl, marker="o", s=60, zorder=5,
                label="validation loss" if i == 0 else None)

plt.xlabel("training step")
plt.ylabel("cross-entropy loss")
plt.title("CoLA fine-tuning")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
logits, y_true = predict(model_cola, cola_eval_loader)
y_pred = logits.argmax(axis=1)

print(f"Validation accuracy : {accuracy_score(y_true, y_pred):.3f}  (majority baseline: {accuracy_score(y_true, np.full_like(y_true, cola_majority)):.3f})")
print(f"Validation MCC      : {matthews_corrcoef(y_true, y_pred):.3f}  (majority baseline: 0.000)")
print()
print(classification_report(y_true, y_pred, labels=[0, 1], target_names=["unacceptable", "acceptable"], zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

plt.figure(figsize=(4.8, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=["unacceptable", "acceptable"],
            yticklabels=["unacceptable", "acceptable"])
plt.xlabel("predicted"); plt.ylabel("true")
plt.title("CoLA — confusion matrix")
plt.tight_layout(); plt.show()

> 💡 MCC and accuracy have different scales, so compare each result with its own task's
> baseline. These scores depend on the sampled training data, seed and checkpoint;
> a small lab run does not establish that one language task is universally harder.

## 🔮 Inspecting grammar predictions

The pairs below contrast acceptable and unacceptable forms in standard English. Such
examples help inspect model behavior; broader acceptability judgments can depend on
context and dialect, and a few predictions do not demonstrate general grammatical understanding.

In [ ]:
grammar_tests = [
    "The judge told the jurors to think carefully.",
    "The judge told that the jurors to think carefully.",
    "The cat sat on the mat.",
    "Cat the on mat sat the.",
    "The dogs run quickly.",
    "The dogs runs quickly.",
]

classify(model_cola, grammar_tests)

# 🎁 Part 7 — Two shortcuts worth knowing

## 💾 Saving and loading your model

`model.save_pretrained` saves model weights and configuration. Save the tokenizer
**separately** with `tokenizer.save_pretrained` into the same directory so the complete
predictor can be reloaded later.

In [ ]:
save_dir = Path("my_sentiment_model")
model_sst2.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# Loading both components back later:
# reloaded = AutoModelForSequenceClassification.from_pretrained(save_dir)
# reloaded_tokenizer = AutoTokenizer.from_pretrained(save_dir)

for saved_file in sorted(save_dir.iterdir()):
    if saved_file.is_file():
        print(f"{saved_file.name:40s} {saved_file.stat().st_size / 1024:.1f} KiB")

In [ ]:
# Our own fine-tuned model can be wrapped in a pipeline, just like the one from Part 1
my_pipeline = pipeline(
    "text-classification",
    model=model_sst2,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

my_pipeline(["What a wonderful evening.", "Total waste of money."],
            truncation=True, max_length=MAX_LEN)

# Activities ✏️

Activity 1 is above (challenging the pipeline). The remaining activities include starter
code and empty cells for your solutions. Use the same data split and seed for comparisons,
and report **validation** results when choosing models or hyperparameters. Keep any final
test set out of those choices.

## ✏️ Activity 2 — Freeze the backbone 🧊

Instead of fine-tuning every weight, set the backbone parameters' `requires_grad` to
`False` and train only the classification head. Our training helper also keeps a fully
frozen backbone in **evaluation mode**: freezing gradients alone does not disable dropout.

**Your task:**

1. Run the starter cell and inspect the trainable parameter count.
2. Train with the same SST-2 subset, epochs and head-initialization seed as the full model.
3. Compare training time and validation accuracy. Does freezing help or hurt in this run?

Frozen features reduce backward-pass work and optimizer state. They still require a
backbone forward pass; the speed and accuracy differences must be measured, not assumed.

In [ ]:
torch.manual_seed(SEED)
model_frozen = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT, num_labels=2, id2label=id2label, label2id=label2id
).to(device)

# Freeze the whole transformer body (everything except the classification head)
for param in model_frozen.base_model.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model_frozen.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_frozen.parameters())
print(f"trainable parameters: {trainable:,} / {total:,}  ({100 * trainable / total:.2f} %)")

## ✏️ Activity 3 — Swap the model 🔄

The Auto classes provide a common interface, but a new checkpoint needs its **own
tokenizer, model and preprocessing**. Changing the string alone does not update objects
that already exist in the notebook kernel.

**Your task:** choose a checkpoint and repeat the SST-2 experiment.

| Checkpoint | Approximate size | Architecture |
|---|---|---|
| `distilbert/distilbert-base-uncased` | 66 M | baseline DistilBERT |
| `bert-base-uncased` | 110 M | BERT-base |
| `roberta-base` | 125 M | RoBERTa-base |
| `google/bert_uncased_L-2_H-128_A-2` | 4 M | two-layer BERT |

1. Set `CHECKPOINT` and reload `tokenizer` with `use_fast=True`.
2. Re-encode the **same raw SST-2 splits** with `encode`; rebuild the collator and loaders.
   The helper retains tokenizer-specific inputs such as BERT's `token_type_ids`.
3. Initialize a fresh sequence classifier with the same label mappings and seed, then train.
4. Compare validation accuracy, parameter count, training time and tokenizer output.

If revisiting the embedding/attention sections, reload **both backbones** too and inspect
the new dimensions. `[CLS]`, WordPiece `##` notation and the illustrated head architecture
are specific to our default model family. Use `model.base_model` for generic freezing.

## ✏️ Activity 4 — How much data do you actually need? 📉

Transfer learning is famously data-efficient. Let's measure how much.

**Your task:** fine-tune SST-2 with `N_TRAIN` = **500**, **2000** and **8000** examples
(1 epoch each, fresh model with the same initialization seed every time). Use prefixes
of the same shuffled training split and keep validation examples fixed. Rebuild the
encoded dataset and loader for each size, then plot **validation accuracy vs. size**.

**Question:** where does the curve start to flatten out? What would that mean if
labelling one example cost you 10 CZK?

## ✏️ Activity 5 — the Pipelines API on IMDB 🎬

The [Pipelines API](https://huggingface.co/docs/transformers/en/main_classes/pipelines)
is a convenient interface for supported tasks and compatible models from the
[Hugging Face model collection](https://huggingface.co/models).

**Your task:**

1. Load the **IMDB** dataset (`load_dataset("stanfordnlp/imdb")`) — full movie reviews, much longer
   than SST-2 sentences.
2. Take a reproducible random sample of **1 000 test reviews** using `shuffle(seed=SEED)`.
3. Classify them with a `pipeline` (you may use
   `distilbert-base-uncased-finetuned-sst-2-english`, or fine-tune your own model).
4. Map predicted label **names** to the dataset's integer labels before computing
   **accuracy** and a **confusion matrix**. The pretrained pipeline returns
   `NEGATIVE`/`POSITIVE`; our custom model uses lowercase names, so normalize them.
5. Print **three reviews the model got wrong** and write one sentence about *why* you
   think it failed.

> ⚠️ IMDB reviews are often longer than 512 tokens — pass `truncation=True` to the
> pipeline (with a supported maximum length and a modest batch size), and consider
> what gets lost by truncation. If you fine-tune on IMDB, use its **training** split
> and reserve the selected test examples for final evaluation.

In [ ]:
imdb = load_dataset("stanfordnlp/imdb")
imdb